In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

orders_df = pd.DataFrame({
    'shop': ['Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town'],
    'weekday': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    'croissants': [99, 79, 89, 80, 73, 112, 112, 81, 66, 91, 67, 75, 126, 118, 95, 71, 85, 90, 97, 111, 123, 79, 98, 87, 92, 78, 98, 135, 81, 86, 85, 86, 93, 122, 116, 56, 89, 68, 77, 71, 117, 121],
    'loaves': [42, 53, 45, 43, 48, 60, 39, 44, 55, 32, 53, 54, 66, 57, 40, 58, 53, 52, 53, 60, 50, 44, 28, 57, 54, 50, 75, 63, 48, 51, 38, 49, 56, 52, 68, 50, 56, 49, 52, 35, 52, 52],
    'cakes': [22, 7, 13, 21, 15, 32, 35, 33, 12, 14, 11, 20, 31, 32, 35, 19, 21, 19, 22, 28, 33, 27, 24, 10, 12, 19, 26, 30, 26, 19, 22, 30, 22, 32, 37, 15, 26, 15, 10, 18, 25, 37],
    'temperature_c': [11.4, 4.9, 6.8, 23.7, 8.2, 19.6, 22.3, 26.5, 4.9, 17.3, 5.4, 4.5, 7.0, 22.8, 26.0, 4.1, 15.9, 13.0, 8.8, 4.6, 12.4, 11.9, 16.1, 10.0, 11.6, 9.2, 10.0, 9.1, 23.3, 21.3, 26.3, 24.8, 12.1, 14.0, 9.5, 17.4, 16.0, 20.7, 6.4, 25.0, 14.4, 16.9],
    'rain_mm': [4.0, 2.8, 2.2, 1.4, 0.0, 2.2, 0.8, 0.0, 0.0, 0.0, 0.2, 4.4, 1.5, 0.0, 2.6, 0.3, 0.0, 0.0, 2.9, 3.2, 5.7, 2.2, 3.8, 0.0, 2.4, 0.0, 0.0, 0.7, 0.0, 0.0, 0.0, 2.7, 0.0, 0.3, 0.0, 4.1, 4.7, 4.5, 0.0, 0.0, 0.0, 0.7],
    'promotion': [True, False, False, False, False, False, False, True, False, False, False, False, False, False, True, False, True, False, True, False, False, False, True, False, False, False, False, True, False, False, False, False, True, False, False, False, True, False, False, False, False, False],
    'complaints': [2, 2, 1, 3, 1, 1, 1, 2, 0, 1, 0, 1, 0, 2, 0, 0, 2, 1, 2, 2, 1, 1, 2, 0, 1, 1, 2, 0, 2, 2, 1, 1, 0, 2, 0, 1, 1, 0, 0, 1, 1, 2],
})

In [ ]:
orders_df["total_items"] = orders_df[["croissants", "loaves", "cakes"]].sum(axis=1)
orders_df["is_weekend"] = orders_df["weekday"].isin(["Sat", "Sun"])

In [ ]:
shop_summary_df = orders_df.groupby("shop").agg(
    days=("weekday", "count"),
    mean_croissants=("croissants", "mean"),
    mean_loaves=("loaves", "mean"),
    mean_cakes=("cakes", "mean"),
    mean_total=("total_items", "mean"),
    rainy_days=("rain_mm", lambda values: int((values > 2).sum())),
    promotion_days=("promotion", "sum"),
).round(2)
weekend_uplift_df = orders_df.groupby(["shop", "is_weekend"])["total_items"].mean().unstack().round(1)
weekend_uplift_df.columns = ["weekday_mean", "weekend_mean"]
weekend_uplift_df["uplift_pct"] = (100 * (weekend_uplift_df["weekend_mean"] / weekend_uplift_df["weekday_mean"] - 1)).round(1)
print(shop_summary_df)
print(weekend_uplift_df)

In [ ]:
for shop_name, shop_rows in orders_df.groupby("shop"):
    busiest_row = shop_rows.loc[shop_rows["total_items"].idxmax()]
    quietest_row = shop_rows.loc[shop_rows["total_items"].idxmin()]
    print(f"{shop_name}: busiest on a {busiest_row['weekday']} with {busiest_row['total_items']} items")
    print(f"{shop_name}: quietest on a {quietest_row['weekday']} with {quietest_row['total_items']} items")
    print(f"{shop_name}: mean complaints per day {shop_rows['complaints'].mean():.2f}")
    print(f"{shop_name}: promotion share {shop_rows['promotion'].mean():.0%}")
rain_correlation = orders_df["rain_mm"].corr(orders_df["croissants"])
temperature_correlation = orders_df["temperature_c"].corr(orders_df["cakes"])
print(f"Rain vs croissants correlation: {rain_correlation:.3f}")
print(f"Temperature vs cakes correlation: {temperature_correlation:.3f}")

In [ ]:
feature_columns = ["croissants", "loaves", "cakes", "temperature_c", "rain_mm"]
feature_matrix = orders_df[feature_columns].to_numpy(dtype=float)
feature_means = feature_matrix.mean(axis=0)
feature_scales = feature_matrix.std(axis=0)
scaled_matrix = (feature_matrix - feature_means) / feature_scales
cluster_count = 3
cluster_rng = np.random.default_rng(11)
centroids = scaled_matrix[cluster_rng.choice(len(scaled_matrix), cluster_count, replace=False)]
for iteration in range(25):
    distances = ((scaled_matrix[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    labels = distances.argmin(axis=1)
    new_centroids = np.array([scaled_matrix[labels == k].mean(axis=0) for k in range(cluster_count)])
    if np.allclose(new_centroids, centroids):
        break
    centroids = new_centroids
orders_df["demand_cluster"] = labels
inertia = float(((scaled_matrix - centroids[labels]) ** 2).sum())
print(f"k-means finished after {iteration + 1} iterations with inertia {inertia:.2f}")

In [ ]:
cluster_profile_df = orders_df.groupby("demand_cluster")[feature_columns + ["total_items"]].mean().round(1)
cluster_profile_df["days"] = orders_df["demand_cluster"].value_counts().sort_index()
cluster_names = {cluster: ("busy" if row["total_items"] >= cluster_profile_df["total_items"].max() else "quiet" if row["total_items"] <= cluster_profile_df["total_items"].min() else "regular") for cluster, row in cluster_profile_df.iterrows()}
orders_df["demand_label"] = orders_df["demand_cluster"].map(cluster_names)
print(cluster_profile_df)
print(orders_df["demand_label"].value_counts().sort_index())

In [ ]:
label_by_shop_df = pd.crosstab(orders_df["shop"], orders_df["demand_label"])
print(label_by_shop_df)

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
for label_name, label_rows in orders_df.groupby("demand_label"):
    axis.scatter(label_rows["temperature_c"], label_rows["total_items"], label=label_name)
axis.set_xlabel("Temperature (C)")
axis.set_ylabel("Items sold")
axis.legend()
plt.show()
for label_name, label_rows in orders_df.groupby("demand_label"):
    print(f"{label_name}: {len(label_rows)} days, mean temperature {label_rows['temperature_c'].mean():.1f} C, mean rain {label_rows['rain_mm'].mean():.1f} mm")